In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization (optional but useful)
import matplotlib.pyplot as plt
import seaborn as sns

# Model
import xgboost as xgb

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

import sagemaker
import boto3
import joblib

In [ ]:
import boto3

BUCKET = "your-s3-bucket"
FILE_PATH = "processed_features.csv"
S3_KEY = "data/processed/processed_features.csv"

s3 = boto3.client("s3", region_name="us-east-1")

s3.upload_file(FILE_PATH, BUCKET, S3_KEY)

print("Upload successful ✅")

In [ ]:
import pandas as pd

s3_path = "your-path"

df = pd.read_csv(s3_path)

print(df.shape)
print(df.head())

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns
print(cat_cols)

In [ ]:
print(df.dtypes)

In [ ]:
print(df.isnull().sum().sum())  # should be 0

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import boto3

# -----------------------------
# Load dataset
# -----------------------------
#df = pd.read_csv("processed_features.csv")

# -----------------------------
# Drop problematic columns
# -----------------------------
cols_to_drop = ['id', 'date_int', 'event_time']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# -----------------------------
# Sort by date (CRITICAL)
# -----------------------------
df = df.sort_values('date').reset_index(drop=True)

# -----------------------------
# Target
# -----------------------------
target = 'sales'

# -----------------------------
# Time-based split (40/10/10/40)
# -----------------------------
n = len(df)
train_end = int(0.4 * n)
val_end   = int(0.5 * n)
test_end  = int(0.6 * n)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:test_end].copy()
prod_df  = df.iloc[test_end:].copy()

# Save metadata BEFORE encoding
test_metadata = test_df[['date', 'store_family_id', 'city', 'state',"store_nbr","store_type"]].copy()
prod_metadata = prod_df[['date', 'store_family_id', 'city', 'state',"store_nbr","store_type"]].copy()
# -----------------------------
# Encode categorical features
# -----------------------------
cat_cols = ["family", "city", "state", "store_type", "store_family_id"]

encoders = {}

for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        val_df[col]   = le.transform(val_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))
        prod_df[col]  = le.transform(prod_df[col].astype(str))
        encoders[col] = le

# -----------------------------
# Separate features and target
# -----------------------------
y_train = train_df[target]
X_train = train_df.drop(columns=[target, 'date'])

y_val = val_df[target]
X_val = val_df.drop(columns=[target, 'date'])

y_test = test_df[target]
X_test = test_df.drop(columns=[target, 'date'])

y_prod = prod_df[target]
X_prod = prod_df.drop(columns=[target, 'date'])

# -----------------------------
# Combine target first (SageMaker format)
# -----------------------------
train_sm = pd.concat([y_train, X_train], axis=1)
val_sm   = pd.concat([y_val, X_val], axis=1)
# -----------------------------
# Prediction files (NO LABEL)
# -----------------------------
# IMPORTANT: enforce same feature order as training
X_test = X_test[X_train.columns]
X_prod = X_prod[X_train.columns]

test_predict = X_test.copy()
prod_predict = X_prod.copy()

# -----------------------------
# Save without header
# -----------------------------
train_sm.to_csv("train_sm.csv", index=False, header=False)
val_sm.to_csv("val_sm.csv", index=False, header=False)

test_predict.to_csv("test_predict.csv", index=False, header=False)
prod_predict.to_csv("prod_predict.csv", index=False, header=False)

# Save labels locally only (for evaluation)
y_test.to_csv("y_test.csv", index=False, header=False)
y_prod.to_csv("y_prod.csv", index=False, header=False)

# -----------------------------
# Upload to S3
# -----------------------------
bucket = "your-s3-bucket"
s3 = boto3.client("s3")

s3.upload_file("train_sm.csv", bucket, "batch_input/train/train_sm.csv")
s3.upload_file("val_sm.csv", bucket, "batch_input/val/val_sm.csv")
# Prediction (NO LABEL)
s3.upload_file("test_predict.csv", bucket, "batch_input/test/test_predict.csv")
s3.upload_file("prod_predict.csv", bucket, "batch_input/prod/prod_predict.csv")

s3.upload_file("y_test.csv", bucket, "batch_input/test/y_test.csv")
s3.upload_file("y_prod.csv", bucket, "batch_input/prod/y_prod.csv")

print("All files uploaded successfully!")

In [ ]:
print("Train feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)
print("Prod feature shape:", X_prod.shape)

In [ ]:
# Save feature names
import json

feature_cols = X_train.columns.tolist()
print(feature_cols)

with open("feature_columns.json", "w") as f:
    json.dump(feature_cols, f)

# Save monitoring datasets WITH headers
train_monitor = pd.concat([y_train, X_train], axis=1)
prod_monitor = X_prod.copy()

train_monitor.to_csv("train_monitor.csv", index=False)
prod_monitor.to_csv("prod_monitor.csv", index=False)

print("Monitoring files created")

In [ ]:
# ============================================================
# NAIVE BASELINE MODEL (Grouped by store_family_id)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# 1. Create safe copy (DO NOT TOUCH MAIN DF)
# ------------------------------------------------------------
df_baseline = df.copy()

# Ensure chronological order
df_baseline = df_baseline.sort_values("date").reset_index(drop=True)

# ------------------------------------------------------------
# 2. Create grouped naive prediction
# Predict yesterday's sales for same store_family_id
# ------------------------------------------------------------
df_baseline["naive_pred"] = (
    df_baseline
    .groupby("store_family_id")["sales"]
    .shift(1)
)

# ------------------------------------------------------------
# 3. Recreate same time split (40 / 10 / 10 / 40)
# ------------------------------------------------------------
n = len(df_baseline)

train_end = int(0.4 * n)
val_end   = int(0.5 * n)
test_end  = int(0.6 * n)

train_baseline = df_baseline.iloc[:train_end].copy()
val_baseline   = df_baseline.iloc[train_end:val_end].copy()

# Drop rows where shift created NaN
val_baseline = val_baseline.dropna(subset=["naive_pred"])

# ------------------------------------------------------------
# 4. Calculate RMSE manually (works in all sklearn versions)
# ------------------------------------------------------------
mse = mean_squared_error(
    val_baseline["sales"],
    val_baseline["naive_pred"]
)

rmse_naive = np.sqrt(mse)

print("====================================")
print("Naive Baseline RMSE:", rmse_naive)
print("====================================")

In [ ]:
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

sess = sagemaker.Session()
role = "<YOUR_SAGEMAKER_ROLE>"
region = "us-east-1"

container = image_uris.retrieve("xgboost", region=region, version="1.7-1")

estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/model_output",
    sagemaker_session=sess
)

estimator.set_hyperparameters(
    objective="reg:squarederror",
    
    # Learning
    num_round=500,              # reduce from 1000
    eta=0.03,                   # slightly lower learning rate
    
    # Tree complexity
    max_depth=4,                # reduce depth
    min_child_weight=5,         # prevent tiny leaf splits
    gamma=1.0,                  # require gain to split
    
    # Sampling
    subsample=0.7,
    colsample_bytree=0.7,
    
    # Regularization
    reg_lambda=5,               # L2 regularization
    reg_alpha=1,                # L1 regularization
    
    # Early stopping
    early_stopping_rounds=30
)


train_input = TrainingInput(f"s3://{bucket}/batch_input/train/train_sm.csv", content_type="text/csv")
val_input   = TrainingInput(f"s3://{bucket}/batch_input/val/val_sm.csv", content_type="text/csv")

estimator.fit({"train": train_input, "validation": val_input})
print("Training feature count:", X_train.shape[1])

In [ ]:
print("Average sales:", y_train.mean())

In [ ]:
transformer = estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/batch_output"
)

transformer.transform(
    data=f"s3://{bucket}/batch_input/test/test_predict.csv",
    content_type="text/csv",
    split_type="Line"
)
transformer.wait()

In [ ]:
bucket = "your-s3-bucket"
key = "batch_output/test_predict.csv.out"  # <- correct key from output_path
output_file = "predictions.csv"

s3 = boto3.client("s3")
s3.download_file(bucket, key, output_file)

print(f"Predictions downloaded to {output_file}")

In [ ]:
import pandas as pd
import numpy as np

# Read predictions
preds = pd.read_csv("predictions.csv", header=None)

# Convert to 1D array
preds = preds.squeeze()  # this turns (n,1) DataFrame into (n,) Series

# Now compute RMSE
rmse = np.sqrt(np.mean((preds.values - y_test.values)**2))
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test.values[:200], label="Actual")
plt.plot(preds.values[:200], label="Predicted")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# If preds is numpy → convert to pandas Series
if isinstance(preds, np.ndarray):
    preds = pd.Series(preds.flatten())

# If preds was loaded from CSV as DataFrame
if isinstance(preds, pd.DataFrame):
    preds = preds.iloc[:, 0]

# Reset index to align
test_metadata = test_metadata.reset_index(drop=True)
preds = preds.reset_index(drop=True)

# Create final planning dataset
final_predictions = test_metadata.copy()
final_predictions["predicted_sales"] = preds

print(final_predictions.head())

In [ ]:
final_predictions.to_csv("retail_planning_predictions.csv", index=False)

s3.upload_file(
    "retail_planning_predictions.csv",
    bucket,
    "final_output/retail_planning_predictions.csv"
)

print("Retail planning file ready!")